# Actividad 2 — La Neurona No Lineal
## Funciones de Activación: ReLU y Leaky ReLU

**Curso:** Machine Learning — Semillero  
**Tema asignado:** ReLU (*Rectified Linear Unit*) y Leaky ReLU

> *"Una neurona sin activación ve el mundo plano. ReLU le enseña a ignorar el ruido y enfocarse en lo que importa."*

---

## 1. ¿Qué son las Funciones de Activación?

Una neurona artificial calcula primero una combinación lineal de sus entradas:

$$z = \mathbf{w}^\top \mathbf{x} + b$$

Sin ningún paso adicional, la salida sería simplemente $\hat{y} = z$, lo que significa que la neurona solo puede aprender **relaciones lineales**. No importa cuántas neuronas apiles, la composición de funciones lineales sigue siendo lineal.

Una **función de activación** $f$ rompe esa limitación al transformar $z$ antes de producir la salida:

$$\hat{y} = f(z)$$

Gracias a $f$, la neurona puede aprender umbrales, saturaciones, asimetrías y cualquier comportamiento no lineal presente en los datos, **sin necesidad de conocer la ecuación matemática subyacente**.

### ¿Para qué sirven?

| Propósito | Explicación |
|-----------|-------------|
| **Romper la linealidad** | Permiten aprender patrones curvos, umbrales y saturaciones |
| **Controlar el rango de salida** | Algunas funciones acotan $\hat{y}$ a $(0,1)$ o $(-1,1)$, útil en clasificación |
| **Habilitar el backpropagation** | La derivada $f'(z)$ es esencial para el cálculo de gradientes |
| **Selección de señales** | ReLU, por ejemplo, "apaga" neuronas que reciben señales negativas |

---

## 2. ReLU — Rectified Linear Unit

### Definición matemática

$$\text{ReLU}(z) = \max(0, z) = \begin{cases} z & \text{si } z > 0 \\ 0 & \text{si } z \leq 0 \end{cases}$$

### Derivada

$$\text{ReLU}'(z) = \mathbf{1}[z > 0] = \begin{cases} 1 & \text{si } z > 0 \\ 0 & \text{si } z \leq 0 \end{cases}$$

La derivada no está definida en $z=0$, pero en la práctica se usa $f'(0) = 0$.

### Rango de salida
$$\hat{y} \in [0, +\infty)$$

---

## 3. Leaky ReLU

### Definición matemática

$$\text{Leaky ReLU}(z) = \max(\alpha z, z) = \begin{cases} z & \text{si } z > 0 \\ \alpha z & \text{si } z \leq 0 \end{cases}$$

donde $\alpha \ll 1$ (valor típico: $0.01$). A diferencia de ReLU, **no mata completamente las neuronas negativas**: les permite pasar una fracción pequeña de la señal.

### Derivada

$$\text{Leaky ReLU}'(z) = \begin{cases} 1 & \text{si } z > 0 \\ \alpha & \text{si } z \leq 0 \end{cases}$$

### Rango de salida
$$\hat{y} \in (-\infty, +\infty)$$

---

## 4. Fortalezas y Errores

### ReLU

| Fortalezas | Errores / Limitaciones |
|------------|------------------------|
| Computacionalmente muy eficiente (solo una comparación) | **Neuronas muertas (*dying ReLU*)**: si $z$ siempre es negativo, el gradiente es 0 y la neurona deja de aprender |
| Gradiente constante en la región activa (evita el problema de gradientes que desaparecen en la región positiva) | No tiene límite superior: puede generar salidas muy grandes |
| Produce representaciones dispersas (muchos ceros), lo que puede actuar como regularización implícita | No centrada en cero: la media de la activación es positiva, lo que puede afectar la dinámica del entrenamiento |
| Convergencia más rápida que sigmoide o tanh en redes profundas | Sensible a la inicialización: con pesos grandes iniciales, muchas neuronas pueden morir desde el inicio |

### Leaky ReLU

| Fortalezas | Errores / Limitaciones |
|------------|------------------------|
| Resuelve el problema de neuronas muertas: el gradiente nunca es exactamente 0 | El valor óptimo de $\alpha$ no es universal: depende del problema |
| Mantiene la eficiencia computacional de ReLU | No es monótona para valores muy negativos cuando $\alpha < 0$ (raramente se usa así) |
| Rango sin restricción, adecuado para regresión general | Para $\alpha$ mal elegido puede comportarse peor que ReLU |

---

## 5. ¿En qué capas se usan?

- **Capas ocultas** de redes neuronales profundas: ReLU y Leaky ReLU son la opción predeterminada en capas intermedias de la mayoría de arquitecturas modernas (MLPs, CNNs, algunos Transformers).
- **Capas convolucionales** (CNN): ReLU es la activación estándar después de cada convolución.
- **Generalmente NO en la capa de salida**: la capa de salida usa activaciones acordes a la tarea (sigmoide para clasificación binaria, softmax para multiclase, identidad para regresión con valores arbitrarios).
- ReLU se usa en la capa de salida **solo cuando se sabe que la salida es no negativa** (por ejemplo, predicción de precios o intensidades).

## 6. Casos de uso

- **ReLU**: redes convolucionales para visión computarizada (ResNet, VGG), redes de lenguaje (capas feed-forward internas), regresión con salidas positivas.
- **Leaky ReLU**: cuando el dataset tiene muchas muestras con activaciones negativas (señales de audio, series temporales con valores negativos), redes generativas adversarias (GAN) donde evitar neuronas muertas es crítico.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, interactive_output, VBox, HBox, Layout
from IPython.display import display

np.random.seed(42)

## 7. Implementación desde cero

Siguiendo el patrón de la Actividad 1, definimos una clase base `ActivationFunction` y derivamos `ReLU` y `LeakyReLU` de ella. Después, la clase `Neuron` acepta cualquier instancia de activación, lo que la hace completamente reutilizable.

In [ ]:
# ── Clase base: todas las funciones de activación heredan de aquí ──
class ActivationFunction:
    """Interfaz común para funciones de activación."""
    def forward(self, z: np.ndarray) -> np.ndarray:
        raise NotImplementedError
    def derivative(self, z: np.ndarray) -> np.ndarray:
        raise NotImplementedError
    def __repr__(self):
        return self.__class__.__name__


# ── ReLU ──
class ReLU(ActivationFunction):
    """
    ReLU(z) = max(0, z)
    Derivada: 1 si z > 0, 0 si z <= 0
    """
    def forward(self, z):
        return np.maximum(0, z)

    def derivative(self, z):
        return (z > 0).astype(float)


# ── Leaky ReLU ──
class LeakyReLU(ActivationFunction):
    """
    Leaky ReLU(z) = z si z > 0, alpha*z si z <= 0
    Derivada: 1 si z > 0, alpha si z <= 0
    """
    def __init__(self, alpha=0.01):
        self.alpha = alpha

    def forward(self, z):
        return np.where(z > 0, z, self.alpha * z)

    def derivative(self, z):
        return np.where(z > 0, 1.0, self.alpha)

    def __repr__(self):
        return f"LeakyReLU(α={self.alpha})"


# ── Identidad (para comparar con Tarea 1) ──
class Identity(ActivationFunction):
    def forward(self, z):    return z
    def derivative(self, z): return np.ones_like(z)


# ── Sigmoide (para referencia) ──
class Sigmoid(ActivationFunction):
    def forward(self, z):
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))
    def derivative(self, z):
        s = self.forward(z)
        return s * (1 - s)


print("Clases de activación definidas correctamente.")

### La clase Neurona

Esta es la misma neurona de la Actividad 1, extendida para aceptar cualquier `ActivationFunction`. Con `activation=Identity()` reproduce exactamente el comportamiento anterior.

In [ ]:
class Neuron:
    """
    Neurona artificial con función de activación intercambiable.
    Modelo: y_hat = f(Xw + b)
    """
    def __init__(self, activation=None, learning_rate=0.01, epochs=1000):
        self.activation    = activation if activation is not None else Identity()
        self.learning_rate = learning_rate
        self.epochs        = epochs
        self.w             = None   # se inicializa al conocer d
        self.b             = 0.0
        self.loss_history  = []

    # ── Propagación hacia adelante ──
    def predict(self, X):
        X = np.atleast_2d(X)
        z = X @ self.w + self.b
        return self.activation.forward(z)

    # ── Entrenamiento con descenso por gradiente ──
    def fit(self, X, y, verbose=False):
        X = np.atleast_2d(X)
        y = np.array(y)
        N, d = X.shape

        # Inicialización pequeña para evitar neuronas muertas en ReLU
        self.w = np.random.randn(d) * 0.01
        self.b = 0.0
        self.loss_history = []

        for epoch in range(self.epochs):
            z      = X @ self.w + self.b
            y_hat  = self.activation.forward(z)
            error  = y_hat - y
            mse    = np.mean(error ** 2)
            self.loss_history.append(mse)

            # Gradientes (regla de la cadena)
            delta  = (2 / N) * error * self.activation.derivative(z)
            dw     = X.T @ delta
            db     = np.sum(delta)

            self.w -= self.learning_rate * dw
            self.b -= self.learning_rate * db

            if verbose and epoch % max(1, self.epochs // 5) == 0:
                print(f"  Época {epoch:5d} | MSE = {mse:.6f}")

        return self

    def mse(self, X, y):
        return np.mean((self.predict(X) - y) ** 2)


# ── Clase visual: hereda de Neuron y agrega gráficas ──
class NeuronVisual(Neuron):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def plot_loss(self, ax=None, label=None, color=None):
        if ax is None:
            fig, ax = plt.subplots(figsize=(7, 4))
        ax.semilogy(self.loss_history, label=label or str(self.activation), color=color)
        ax.set_xlabel("Épocas")
        ax.set_ylabel("MSE (escala log)")
        ax.set_title("Curva de pérdida")
        ax.legend()
        ax.grid(True, alpha=0.3)

    def plot_fit(self, X_train, y_train, X_test, y_test, ax=None, label=None):
        if ax is None:
            fig, ax = plt.subplots(figsize=(7, 4))
        x_line = np.linspace(X_train.min(), X_train.max(), 300).reshape(-1, 1)
        y_line = self.predict(x_line)
        ax.scatter(X_train, y_train, s=15, alpha=0.5, label="Entrenamiento")
        ax.scatter(X_test,  y_test,  s=20, alpha=0.7, marker='^', label="Prueba", color='orange')
        ax.plot(x_line, y_line, linewidth=2, label=label or str(self.activation))
        ax.legend()
        ax.grid(True, alpha=0.3)


print("Clases Neuron y NeuronVisual definidas.")

## 8. Visualización de ReLU y Leaky ReLU

In [ ]:
z_vals = np.linspace(-3, 3, 500)

relu      = ReLU()
lrelu_01  = LeakyReLU(alpha=0.01)
lrelu_02  = LeakyReLU(alpha=0.2)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Función
ax = axes[0]
ax.plot(z_vals, relu.forward(z_vals),     label='ReLU',             color='royalblue', lw=2)
ax.plot(z_vals, lrelu_01.forward(z_vals), label='Leaky ReLU α=0.01', color='tomato',    lw=2)
ax.plot(z_vals, lrelu_02.forward(z_vals), label='Leaky ReLU α=0.2',  color='seagreen',  lw=2, ls='--')
ax.axhline(0, color='gray', lw=0.8)
ax.axvline(0, color='gray', lw=0.8)
ax.set_title('Función f(z)', fontsize=13)
ax.set_xlabel('z');  ax.set_ylabel('f(z)')
ax.legend();  ax.grid(True, alpha=0.3)

# Derivada
ax = axes[1]
ax.plot(z_vals, relu.derivative(z_vals),     label="ReLU'",              color='royalblue', lw=2)
ax.plot(z_vals, lrelu_01.derivative(z_vals), label="Leaky ReLU' α=0.01", color='tomato',    lw=2)
ax.plot(z_vals, lrelu_02.derivative(z_vals), label="Leaky ReLU' α=0.2",  color='seagreen',  lw=2, ls='--')
ax.axhline(0, color='gray', lw=0.8)
ax.set_title("Derivada f'(z)", fontsize=13)
ax.set_xlabel('z');  ax.set_ylabel("f'(z)")
ax.legend();  ax.grid(True, alpha=0.3)
ax.set_ylim(-0.1, 1.3)

plt.suptitle('ReLU vs Leaky ReLU: función y derivada', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Datos de prueba: fenómenos con umbral

ReLU y Leaky ReLU son especialmente adecuadas para fenómenos que presentan un **umbral claro**: la salida es prácticamente cero hasta que la entrada supera un punto, y luego crece. Esto imita el comportamiento de neuronas biológicas y de muchos procesos físicos (corriente en un diodo, activación enzimática, etc.).

Generamos tres conjuntos de datos para explorar cómo se comportan las funciones.

In [ ]:
np.random.seed(42)
N = 400
x_all = np.random.uniform(-6, 6, N)
idx   = np.argsort(x_all)
x_all = x_all[idx]

# D1 — Sigmoidal (saturación)
A, k, x0 = 3.0, 1.5, 0.5
y_D1_true = A / (1 + np.exp(-k * (x_all - x0)))
y_D1      = y_D1_true + np.random.normal(0, 0.15, N)

# D2 — Umbral tipo ReLU (muy adecuado para esta función)
y_D2_true = np.maximum(0, x_all)          # ¡la función verdadera ES ReLU!
y_D2      = y_D2_true + np.random.normal(0, 0.3, N)

# D3 — Libre: umbral asimétrico  
y_D3_true = np.where(x_all > 0, 1.5 * np.sqrt(np.abs(x_all)), 0.1 * x_all)
y_D3      = y_D3_true + np.random.normal(0, 0.2, N)

# División 80/20
split = int(0.8 * N)
X_tr, X_te = x_all[:split].reshape(-1,1), x_all[split:].reshape(-1,1)

def get_sets(y):
    return y[:split], y[split:]

y_D1_tr, y_D1_te = get_sets(y_D1)
y_D2_tr, y_D2_te = get_sets(y_D2)
y_D3_tr, y_D3_te = get_sets(y_D3)

# Visualización de los tres datasets
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (y, yt, name) in zip(axes, [
        (y_D1, y_D1_true, 'D1 — Sigmoidal'),
        (y_D2, y_D2_true, 'D2 — Umbral / ReLU verdadera'),
        (y_D3, y_D3_true, 'D3 — Umbral asimétrico')]):
    ax.scatter(x_all, y,  s=8, alpha=0.4, label='Datos con ruido')
    ax.plot(x_all, yt, color='red', lw=2, label='Curva verdadera')
    ax.set_title(name, fontsize=11)
    ax.set_xlabel('x');  ax.legend(fontsize=8);  ax.grid(True, alpha=0.3)

plt.suptitle('Conjuntos de datos para la Actividad 2', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Comparación de activaciones sobre los tres datasets

In [ ]:
ACTIVATIONS = [
    Identity(),
    Sigmoid(),
    ReLU(),
    LeakyReLU(alpha=0.01),
    LeakyReLU(alpha=0.1),
]
COLORS  = ['gray', 'purple', 'royalblue', 'tomato', 'seagreen']
DATASETS = [
    ('D1 — Sigmoidal',          y_D1_tr, y_D1_te),
    ('D2 — Umbral / ReLU',      y_D2_tr, y_D2_te),
    ('D3 — Umbral asimétrico',  y_D3_tr, y_D3_te),
]

results = {}  # Para la tabla resumen

fig, axes = plt.subplots(3, 2, figsize=(14, 15))

for row, (dname, y_tr, y_te) in enumerate(DATASETS):
    ax_loss = axes[row, 0]
    ax_fit  = axes[row, 1]

    ax_loss.set_title(f'{dname} — Curva de pérdida', fontsize=11)
    ax_fit.set_title(f'{dname} — Ajuste del modelo', fontsize=11)

    ax_fit.scatter(X_tr, y_tr, s=8,  alpha=0.4, color='steelblue', label='Entrenamiento')
    ax_fit.scatter(X_te, y_te, s=15, alpha=0.7, color='orange',    label='Prueba', marker='^')

    x_line = np.linspace(-6, 6, 400).reshape(-1, 1)

    for act, col in zip(ACTIVATIONS, COLORS):
        neu = NeuronVisual(activation=act, learning_rate=0.005, epochs=3000)
        neu.fit(X_tr, y_tr)
        mse_test  = neu.mse(X_te, y_te)
        label     = str(act)
        results[(dname, label)] = mse_test

        ax_loss.semilogy(neu.loss_history, label=label, color=col, lw=1.5)
        ax_fit.plot(x_line, neu.predict(x_line), label=label, color=col, lw=2)

    ax_loss.set_xlabel('Épocas'); ax_loss.set_ylabel('MSE'); ax_loss.legend(fontsize=7); ax_loss.grid(True, alpha=0.3)
    ax_fit.legend(fontsize=7);  ax_fit.grid(True, alpha=0.3)

plt.suptitle('Comparación de funciones de activación', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Tabla resumen de MSE en prueba

In [ ]:
import pandas as pd

rows = {}
for (dname, aname), mse_val in results.items():
    if dname not in rows:
        rows[dname] = {}
    rows[dname][aname] = round(mse_val, 5)

df_results = pd.DataFrame(rows).T
print("MSE en conjunto de prueba por activación y dataset:\n")
display(df_results.style.highlight_min(axis=1, color='lightgreen').format("{:.5f}"))

## 11. Widget interactivo — Explorador de hiperparámetros

Ajusta en tiempo real la función de activación, la tasa de aprendizaje, el número de épocas y el nivel de ruido. El panel muestra simultáneamente la función de activación, el ajuste sobre los datos y la curva de pérdida.

In [ ]:
# ── Widgets ──
w_act   = widgets.Dropdown(
    options=['ReLU', 'Leaky ReLU (α=0.01)', 'Leaky ReLU (α=0.1)', 'Leaky ReLU (α=0.2)', 'Sigmoide', 'Identidad'],
    value='ReLU', description='Activación:', style={'description_width': 'initial'})

w_lr    = widgets.FloatLogSlider(value=0.005, base=10, min=-4, max=0, step=0.1,
    description='Tasa α:', readout_format='.4f', style={'description_width': 'initial'})

w_ep    = widgets.IntSlider(value=2000, min=100, max=8000, step=100,
    description='Épocas:', style={'description_width': 'initial'})

w_noise = widgets.FloatSlider(value=0.2, min=0.0, max=1.5, step=0.05,
    description='Ruido σ:', style={'description_width': 'initial'})

w_data  = widgets.Dropdown(
    options=['D1 — Sigmoidal', 'D2 — Umbral / ReLU', 'D3 — Umbral asimétrico'],
    value='D2 — Umbral / ReLU', description='Dataset:', style={'description_width': 'initial'})

w_alpha = widgets.FloatSlider(value=0.01, min=0.0, max=0.5, step=0.01,
    description='α Leaky:', style={'description_width': 'initial'})

out = widgets.Output()

def get_activation(name, alpha_leaky):
    mapping = {
        'ReLU':               ReLU(),
        'Leaky ReLU (α=0.01)': LeakyReLU(0.01),
        'Leaky ReLU (α=0.1)':  LeakyReLU(0.1),
        'Leaky ReLU (α=0.2)':  LeakyReLU(0.2),
        'Sigmoide':           Sigmoid(),
        'Identidad':          Identity(),
    }
    return mapping[name]

def build_dataset(dname, noise):
    np.random.seed(42)
    x = np.random.uniform(-6, 6, 400)
    x = np.sort(x)
    if 'D1' in dname:
        y_true = 3.0 / (1 + np.exp(-1.5 * (x - 0.5)))
    elif 'D2' in dname:
        y_true = np.maximum(0, x)
    else:
        y_true = np.where(x > 0, 1.5 * np.sqrt(np.abs(x)), 0.1 * x)
    y = y_true + np.random.normal(0, noise, len(x))
    split = int(0.8 * len(x))
    X_tr = x[:split].reshape(-1, 1);  y_tr = y[:split]
    X_te = x[split:].reshape(-1, 1);  y_te = y[split:]
    return X_tr, y_tr, X_te, y_te, x, y_true

def update(change=None):
    with out:
        out.clear_output(wait=True)
        act     = get_activation(w_act.value, w_alpha.value)
        X_tr, y_tr, X_te, y_te, x_all, y_true = build_dataset(w_data.value, w_noise.value)

        neu = NeuronVisual(activation=act, learning_rate=w_lr.value, epochs=w_ep.value)
        neu.fit(X_tr, y_tr)
        mse_te = neu.mse(X_te, y_te)

        fig, axes = plt.subplots(1, 3, figsize=(16, 4))

        # Panel 1: función de activación
        z_v = np.linspace(-4, 4, 400)
        axes[0].plot(z_v, act.forward(z_v),     label='f(z)',   lw=2.5, color='royalblue')
        axes[0].plot(z_v, act.derivative(z_v),  label="f'(z)",  lw=2,   color='tomato', ls='--')
        axes[0].axhline(0, color='gray', lw=0.8); axes[0].axvline(0, color='gray', lw=0.8)
        axes[0].set_title(f'Activación: {act}', fontsize=11)
        axes[0].legend(); axes[0].grid(True, alpha=0.3)

        # Panel 2: ajuste
        x_line = np.linspace(-6, 6, 400).reshape(-1, 1)
        axes[1].scatter(X_tr, y_tr, s=10, alpha=0.4, label='Entrenamiento', color='steelblue')
        axes[1].scatter(X_te, y_te, s=20, alpha=0.7, marker='^', label='Prueba', color='orange')
        axes[1].plot(x_all, y_true, color='black', lw=1.5, ls=':', label='Curva verdadera')
        axes[1].plot(x_line, neu.predict(x_line), color='tomato', lw=2.5, label='Modelo')
        axes[1].set_title(f'Ajuste del modelo | MSE prueba = {mse_te:.4f}', fontsize=11)
        axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

        # Panel 3: curva de pérdida
        axes[2].semilogy(neu.loss_history, color='royalblue', lw=1.5)
        axes[2].set_xlabel('Épocas'); axes[2].set_ylabel('MSE (log)')
        axes[2].set_title(f'Pérdida final = {neu.loss_history[-1]:.5f}', fontsize=11)
        axes[2].grid(True, alpha=0.3)

        # Parámetros aprendidos
        fig.suptitle(
            f'w = {neu.w[0]:.4f}  |  b = {neu.b:.4f}  |  α = {w_lr.value:.5f}  |  Épocas = {w_ep.value}',
            fontsize=11
        )
        plt.tight_layout()
        plt.show()

# Conectar widgets
for w in [w_act, w_lr, w_ep, w_noise, w_data, w_alpha]:
    w.observe(update, names='value')

controls = VBox([
    HBox([w_act,   w_data]),
    HBox([w_lr,    w_ep]),
    HBox([w_noise, w_alpha]),
])

display(controls, out)
update()  # primera ejecución

## 12. Widget interactivo — Problema del dying ReLU

Cuando la tasa de aprendizaje es muy alta, los pesos se actualizan tanto que $z$ siempre es negativo. La derivada de ReLU es 0 para $z \leq 0$, por lo que los gradientes se vuelven todos cero y el modelo deja de aprender por completo. Leaky ReLU evita esto porque su derivada es $\alpha > 0$ incluso cuando $z < 0$.

Usa el slider para reproducir y observar el colapso:

In [ ]:
w_lr_dead   = widgets.FloatLogSlider(value=0.5, base=10, min=-3, max=1, step=0.1,
    description='Tasa α (dying ReLU demo):', readout_format='.4f',
    style={'description_width': 'initial'}, layout=Layout(width='550px'))

out_dead = widgets.Output()

def dying_relu_demo(change=None):
    with out_dead:
        out_dead.clear_output(wait=True)
        np.random.seed(42)
        X  = np.random.uniform(-6, 6, 300).reshape(-1, 1)
        y  = np.maximum(0, X.ravel()) + np.random.normal(0, 0.3, 300)

        relu_n   = Neuron(activation=ReLU(),           learning_rate=w_lr_dead.value, epochs=2000)
        lrelu_n  = Neuron(activation=LeakyReLU(0.01),  learning_rate=w_lr_dead.value, epochs=2000)
        relu_n.fit(X, y)
        lrelu_n.fit(X, y)

        fig, axes = plt.subplots(1, 2, figsize=(13, 4))
        axes[0].semilogy(relu_n.loss_history,  label='ReLU',          color='royalblue', lw=2)
        axes[0].semilogy(lrelu_n.loss_history, label='Leaky ReLU 0.01', color='tomato', lw=2)
        axes[0].set_xlabel('Épocas'); axes[0].set_ylabel('MSE (log)')
        axes[0].set_title(f'Curva de pérdida — α={w_lr_dead.value:.4f}', fontsize=11)
        axes[0].legend(); axes[0].grid(True, alpha=0.3)

        x_line = np.linspace(-6, 6, 300).reshape(-1, 1)
        axes[1].scatter(X, y, s=10, alpha=0.4, color='gray', label='Datos')
        axes[1].plot(x_line, relu_n.predict(x_line),   label='ReLU',       color='royalblue', lw=2.5)
        axes[1].plot(x_line, lrelu_n.predict(x_line),  label='Leaky ReLU', color='tomato',    lw=2.5)
        axes[1].plot(x_line, np.maximum(0, x_line),    label='Verdadera',  color='black', lw=1.5, ls='--')
        axes[1].set_title('Ajuste del modelo', fontsize=11)
        axes[1].legend(); axes[1].grid(True, alpha=0.3)

        relu_final  = relu_n.loss_history[-1]
        lrelu_final = lrelu_n.loss_history[-1]
        fig.suptitle(
            f'ReLU MSE final: {relu_final:.5f}  |  Leaky ReLU MSE final: {lrelu_final:.5f}',
            fontsize=11
        )
        plt.tight_layout()
        plt.show()
        if relu_final > lrelu_final * 5:
            print(">> ¡Se observa dying ReLU! Leaky ReLU sigue aprendiendo mientras ReLU colapsa.")
        else:
            print(">> Ambas funciones convergen. Aumenta α para ver el colapso de ReLU.")

w_lr_dead.observe(dying_relu_demo, names='value')
display(w_lr_dead, out_dead)
dying_relu_demo()

## 13. Gradient Check numérico

Verificamos que las derivadas implementadas son correctas comparándolas con la derivada numérica por diferencias finitas:

$$f'(z) \approx \frac{f(z+\epsilon) - f(z-\epsilon)}{2\epsilon}$$

In [ ]:
eps = 1e-5
z_test = np.array([-2.0, -1.0, -0.5, 0.5, 1.0, 2.0])  # evitar z=0 exacto

print(f"{'Función':<20} {'z':>6} {'Analítica':>12} {'Numérica':>12} {'Error abs':>12}")
print("-" * 65)

for act in [ReLU(), LeakyReLU(0.01), LeakyReLU(0.1), Sigmoid()]:
    for z in z_test:
        analytical = act.derivative(np.array([z]))[0]
        numerical  = (act.forward(np.array([z + eps])) - act.forward(np.array([z - eps])))[0] / (2 * eps)
        print(f"{str(act):<20} {z:>6.2f} {analytical:>12.6f} {numerical:>12.6f} {abs(analytical - numerical):>12.2e}")
    print()

## 14. Análisis y Reflexión

### R1 — Función de activación vs. linealización de características
En la Tarea 1 se usó la transformación $x \rightarrow [x, x^2]^\top$ para capturar curvatura. Con ReLU o Leaky ReLU la neurona puede aproximar funciones no lineales **sin** necesidad de diseñar esa transformación manualmente. Sin embargo, una sola neurona con ReLU produce funciones lineales a trozos: es complementaria a la linealización cuando se necesita capturar saturaciones simétricas (para eso sigmoide/tanh son mejores).

### R2 — Dying ReLU vs. convergencia de Leaky ReLU
Con tasas de aprendizaje altas (por encima de ~0.5 en los datos de esta práctica), ReLU colapsa porque los pesos se actualizan tanto que $z = wx + b$ siempre es negativo para todo el dataset, haciendo $f'(z) = 0$ y deteniendo el aprendizaje. Leaky ReLU mantiene $\alpha > 0$ en la región negativa, por lo que los gradientes nunca son exactamente cero y el entrenamiento continúa.

### R3 — Dataset D3 (libre): revelación
La ecuación verdadera de D3 era:
$$y = \begin{cases} 1.5\sqrt{x} & \text{si } x > 0 \\ 0.1 \cdot x & \text{si } x \leq 0 \end{cases}$$
Es un umbral asimétrico con crecimiento sublineal para $x > 0$. Leaky ReLU se acercó más que ReLU pura porque la región $x < 0$ tiene pendiente negativa (no cero), lo que Leaky ReLU puede representar parcialmente con $\alpha > 0$.

### R4 — Limitaciones de una sola neurona
Una sola neurona activada produce una **función lineal a trozos** (ReLU) o una función sigmoidal simple. Para reconocer imágenes o procesar lenguaje natural se necesita aprender jerarquías de características (bordes → texturas → objetos), lo que requiere **múltiples capas**. Cada capa compone no linearidades sobre las representaciones de la capa anterior, permitiendo que la red construya conceptos progresivamente más abstractos. Eso es exactamente lo que habilitan las redes neuronales profundas.

---

## 15. Conclusión

ReLU y Leaky ReLU son las funciones de activación más usadas en capas ocultas de redes modernas por su eficiencia computacional y porque evitan el problema de gradientes que desaparecen presente en sigmoide y tanh. La principal debilidad de ReLU es el **dying ReLU**: cuando muchas neuronas reciben activaciones negativas persistentes, sus gradientes se vuelven cero y dejan de aprender. Leaky ReLU corrige esto al permitir un gradiente pequeño pero distinto de cero en la región negativa.

La elección entre ambas depende del problema: para datos con umbral claro y salidas no negativas, ReLU es suficiente; cuando el dataset tiene señales negativas importantes o se está trabajando con redes generativas, Leaky ReLU (o sus variantes como PReLU y ELU) es preferible.